# Regularized Matrix Factorization

This collaborative filtering technique predicts user preferences by decomposing a spare user-item interaction matrix into lower-dimenssional, dense user and item latent factor matrices. It adds regularization terms to the loss function to penalize large factor values, preventing overfitting and improving generalization on unseen data.

### 1. Load Dataset

In [5]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import mean_squared_error
from math import sqrt


In [6]:
movies = pd.read_csv("../data/raw/movies_metadata.csv", low_memory=False)
keywords = pd.read_csv("../data/raw/keywords.csv")
credits = pd.read_csv("../data/raw/credits.csv")
links = pd.read_csv("../data/raw/links.csv")
links_small = pd.read_csv("../data/raw/links_small.csv")


df = pd.read_csv("../data/raw/ratings_small.csv")

df.info()
df.head()

<class 'pandas.DataFrame'>
RangeIndex: 100004 entries, 0 to 100003
Data columns (total 4 columns):
 #   Column     Non-Null Count   Dtype  
---  ------     --------------   -----  
 0   userId     100004 non-null  int64  
 1   movieId    100004 non-null  int64  
 2   rating     100004 non-null  float64
 3   timestamp  100004 non-null  int64  
dtypes: float64(1), int64(3)
memory usage: 3.1 MB


,userId,movieId,rating,timestamp
0,1,31,2.5,1260759144
1,1,1029,3.0,1260759179
2,1,1061,3.0,1260759182
3,1,1129,2.0,1260759185
4,1,1172,4.0,1260759205


### 2. Clean data, split into train/test (based on timestamp)

We want to drop sparse users, then base the train/test split on the timestamp a user has watched the movie. 

In [12]:
# drop users with less than 5 ratings
counts = df["userId"].value_counts()
df = df[df['userId'].isin(counts[counts >= 5].index)].copy()

# split into train and test sets
df.sort_values(['userId', 'timestamp'], inplace=True)

train, test = [], []
for userID, group in df.groupby('userId'):
    cut = int(len(group) * 0.8)
    train.append(group.iloc[:cut])
    test.append(group.iloc[cut:])
train_set = pd.concat(train).reset_index(drop=True)
test_set = pd.concat(test).reset_index(drop=True)

print(f"\nTrain: {len(train_set):,} | Test: {len(test_set):,}")



Train: 79,748 | Test: 20,256


### 3. Index encoding

Since the movie/user IDs are arbitrary numbers, we want to map them to contiguous integer indices after filtering through each entry

In [16]:
all_users = df['userId'].unique()
all_movies = df['movieId'].unique()

user2index = {u: i for i, u in enumerate(all_users)}
movie2index = {m: i for i, m in enumerate(all_movies)}

n_users = len(all_users)
n_movies = len(all_movies)
print(f"n_users={n_users}  n_movies={n_movies}")

def encode(df_split):
    mask = df_split['movieId'].isin(movie2index)
    d = df_split[mask]
    u = d['userId'].map(user2index).values
    m = d['movieId'].map(movie2index).values
    r = d['rating'].values.astype(np.float32)
    return u, m, r

u_tr, m_tr, r_tr = encode(train_set)
u_te, m_te, r_te = encode(test_set)
print(f"Train triples: {len(r_tr):,}  |  Test triples: {len(r_te):,}")


<class 'dict'>
n_users=671  n_movies=9066
Train triples: 79,748  |  Test triples: 20,256


### 4. Model
Regularized matrix factorization trained with SGD

**Objective**
$$
\min_{P,Q,b_u,b_i}\sum_{(u,i)\in\mathcal{K}}\!\left(r_{ui} - \mu - b_u - b_i - p_u^\top q_i\right)^2
+ \lambda\!\left(\|p_u\|^2+\|q_i\|^2+b_u^2+b_i^2\right)
$$

| Symbol | Meaning |
|--------|--------|
| $P\in\mathbb{R}^{n_u\times k}$ | User latent matrix |
| $Q\in\mathbb{R}^{n_i\times k}$ | Item latent matrix |
| $b_u, b_i$ | User / item bias |
| $\mu$ | Global mean |
| $\lambda$ | L2 regularization coefficient |


In [17]:
class MatrixFactorization:
    def __init__(self, n_users, n_items, n_factors=20, lr=0.005, reg=0.02, n_epochs=20, seed=42):
        self.n_factors = n_factors
        self.lr        = lr
        self.reg       = reg
        self.n_epochs  = n_epochs

        rng = np.random.default_rng(seed)
        scale = 0.1
        self.P  = rng.normal(0, scale, (n_users, n_factors)).astype(np.float32)   # user factors
        self.Q  = rng.normal(0, scale, (n_items, n_factors)).astype(np.float32)   # item factors
        self.bu = np.zeros(n_users, dtype=np.float32)   # user biases
        self.bi = np.zeros(n_items, dtype=np.float32)   # item biases
        self.mu = 0.0                                    # global mean (set at fit time)


    # prediction
    def predict(self, u, m):
        """Vectorised: u, m are integer arrays."""
        return self.mu + self.bu[u] + self.bi[m] + np.sum(self.P[u] * self.Q[m], axis=1)
    
    # training 
    def fit(self, u_tr, m_tr, r_tr, u_te=None, m_te=None, r_te=None, verbose=True):
        self.mu = r_tr.mean()
        n       = len(r_tr)
        idx     = np.arange(n)

        self.train_rmse_hist = []
        self.test_rmse_hist  = []

        for epoch in range(1, self.n_epochs + 1):
            np.random.shuffle(idx)      # in-place shuffle for SGD

            for i in idx:
                u, m, r = u_tr[i], m_tr[i], r_tr[i]
                err = r - (self.mu + self.bu[u] + self.bi[m] + self.P[u] @ self.Q[m])

                # gradient step
                self.bu[u] += self.lr * (err - self.reg * self.bu[u])
                self.bi[m] += self.lr * (err - self.reg * self.bi[m])
                pu_old      = self.P[u].copy()
                self.P[u]  += self.lr * (err * self.Q[m] - self.reg * self.P[u])
                self.Q[m]  += self.lr * (err * pu_old    - self.reg * self.Q[m])

            # epoch metrics
            tr_rmse = sqrt(mean_squared_error(r_tr, self.predict(u_tr, m_tr)))
            self.train_rmse_hist.append(tr_rmse)

            if u_te is not None:
                te_rmse = sqrt(mean_squared_error(r_te, self.predict(u_te, m_te)))
                self.test_rmse_hist.append(te_rmse)

            if verbose and epoch % 5 == 0:
                msg = f"Epoch {epoch:>3}/{self.n_epochs}  train RMSE={tr_rmse:.4f}"
                if u_te is not None:
                    msg += f"  test RMSE={te_rmse:.4f}"
                print(msg)

        return self


### 5. Train

In [18]:
model = MatrixFactorization(
    n_users=n_users,
    n_items=n_movies,
    n_factors=20,
    lr=0.005,
    reg=0.02,
    n_epochs=30,
)

model.fit(u_tr, m_tr, r_tr, u_te, m_te, r_te)

Epoch   5/30  train RMSE=0.8799  test RMSE=0.9287
Epoch  10/30  train RMSE=0.8496  test RMSE=0.9178
Epoch  15/30  train RMSE=0.8228  test RMSE=0.9146
Epoch  20/30  train RMSE=0.7881  test RMSE=0.9129
Epoch  25/30  train RMSE=0.7452  test RMSE=0.9130
Epoch  30/30  train RMSE=0.7005  test RMSE=0.9147


### 6. Evaluation

In [19]:
def rmse(true, pred): return sqrt(mean_squared_error(true, pred))
def mae(true, pred):  return np.mean(np.abs(true - pred))

train_preds = model.predict(u_tr, m_tr)
test_preds  = model.predict(u_te, m_te)

# Clip predictions to valid rating range
rating_min, rating_max = df['rating'].min(), df['rating'].max()
test_preds_clipped = np.clip(test_preds, rating_min, rating_max)

print("=" * 40)
print(f"Train RMSE : {rmse(r_tr, train_preds):.4f}")
print(f"Test  RMSE : {rmse(r_te, test_preds):.4f}  (raw)")
print(f"Test  RMSE : {rmse(r_te, test_preds_clipped):.4f}  (clipped to [{rating_min},{rating_max}])")
print(f"Test  MAE  : {mae(r_te, test_preds_clipped):.4f}")
print("=" * 40)

Train RMSE : 0.7005
Test  RMSE : 0.9147  (raw)
Test  RMSE : 0.9143  (clipped to [0.5,5.0])
Test  MAE  : 0.6983


### 7. Adjust hyperparams

### 8. Top-N recommendations

In [21]:
def recommend(model, user_raw_id, n=10, seen_movie_ids=None):
    """
    Return top-N movie raw IDs predicted for `user_raw_id`.
    Excludes movies the user already rated (seen_movie_ids).
    """
    u_idx = user2index[user_raw_id]
    all_m = np.arange(n_movies)
    preds = model.predict(np.full(n_movies, u_idx), all_m)

    if seen_movie_ids:
        seen_idx = [movie2index[m] for m in seen_movie_ids if m in movie2index]
        preds[seen_idx] = -np.inf

    top_idx = np.argsort(preds)[::-1][:n]
    idx2movie = {v: k for k, v in movie2index.items()}
    return [(idx2movie[i], round(preds[i], 3)) for i in top_idx]

sample_user = all_users[0]
seen = set(train_set[train_set['userId'] == sample_user]['movieId'])

print(f"Top-10 recommendations for user {sample_user}:")
for rank, (movie_id, score) in enumerate(recommend(model, sample_user, n=10, seen_movie_ids=seen), 1):
    print(f"  {rank:>2}. movieId={movie_id}  predicted rating={score}")

Top-10 recommendations for user 1:
   1. movieId=2318  predicted rating=3.694000005722046
   2. movieId=318  predicted rating=3.686000108718872
   3. movieId=307  predicted rating=3.6540000438690186
   4. movieId=969  predicted rating=3.63700008392334
   5. movieId=1945  predicted rating=3.555000066757202
   6. movieId=1221  predicted rating=3.5439999103546143
   7. movieId=3730  predicted rating=3.5360000133514404
   8. movieId=1203  predicted rating=3.5290000438690186
   9. movieId=50  predicted rating=3.5179998874664307
  10. movieId=306  predicted rating=3.513000011444092
